In [13]:
%cd /content
!rm -rf vnlegal-rag-v2
!git clone https://{GH_TOKEN}@github.com/hyperformancelabs/vnlegal-rag-v2.git
%cd vnlegal-rag-v2
!pip install faiss-cpu
!pip install -e .

/content
Cloning into 'vnlegal-rag-v2'...
remote: Enumerating objects: 304, done.
remote: Counting objects: 100% (196/196), done.
remote: Compressing objects: 100% (84/84), done.
remote: Total 304 (delta 117), reused 174 (delta 99), pack-reused 108 (from 2)
Receiving objects: 100% (304/304), 121.22 MiB | 13.91 MiB/s, done.
Resolving deltas: 100% (148/148), done.
/content/vnlegal-rag-v2
Obtaining file:///content/vnlegal-rag-v2
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for vnlegal-rag-v2 (pyproject.toml) ... done
  Created wheel for vnlegal-rag-v2: filename=vnlegal_rag_v2-0.1.0-0.editable-py3-none-any.whl size=1381 sha256=03e64cf7e36bf627c2322e8f416907e53192451025634fbc197fb06baa6c2a0d
  Stored in directory: /tmp/pip-ephem-wheel-cache-9460dvaj/wheels/86/8f/16/c3124b4be639db7d2b065a5b714f123859ed3325b43934

In [14]:
!git pull

Already up to date.


In [15]:
%cd data/raw
!cat dataset_part_* > dataset.7z
!7z x dataset.7z
%cd ../..
!ls data/raw/Train

/content/vnlegal-rag-v2/data/raw

7-Zip [64] 16.02 : Copyright (c) 1999-2016 Igor Pavlov : 2016-05-21
p7zip Version 16.02 (locale=en_US.UTF-8,Utf16=on,HugeFiles=on,64 bits,12 CPUs Intel(R) Xeon(R) CPU @ 2.20GHz (50657),ASM,AES-NI)

Scanning the drive for archives:
  0M Sca        1 file, 126901382 bytes (122 MiB)

Extracting archive: dataset.7z
--
Path = dataset.7z
Type = 7z
Physical Size = 126901382
Headers Size = 346
Method = LZMA2:26
Solid = +
Blocks = 1

      1% 3 - Train/corpus.cs                          3% 3 - Train/corpus.cs                          5% 3 - Train/corpus.cs                          7% 3 - Train/corpus.cs                          8% 3 - Train/corpus.cs                         10% 3 - Train/corpus.cs                         12% 3 - Train/corpus.cs                         13% 3 - Train/corpus.cs                         15% 3 - Train/corpus.cs                         17% 3 - Train/corpus.cs                         18% 3 - Train/corpus.cs                         20% 

In [16]:
!pip install -U huggingface_hub
!hf auth login --token {HF_TOKEN}

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: fineGrained).
The token `base` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `base`


In [17]:
%%writefile configs/data/default.yaml
raw_path: data/raw
processed_path: data/processed
eval_size: 0.2
random_state: 36
segmentation: null
overwrite: true

Overwriting configs/data/default.yaml


In [18]:
!python scripts/process_data.py --config configs/data/default.yaml

Segmenting QA pairs: 100% 133568/133568 [00:00<00:00, 346644.16it/s]
Data processing completed. Saved to data/processed


In [19]:
!python scripts/update_batch_size.py 32 

  configs/model-selection/embeddinggemma-300m.yaml: batch_size=32 (updated)
  configs/model-selection/f2llm-v2-160m.yaml: batch_size=32 (updated)
  configs/model-selection/f2llm-v2-330m.yaml: batch_size=32 (updated)
  configs/model-selection/granite-embedding-311m-multilingual-r2.yaml: batch_size=32 (updated)
  configs/model-selection/granite-embedding-97m-multilingual-r2.yaml: batch_size=32 (updated)
  configs/model-selection/harrier-oss-v1-270m.yaml: batch_size=32 (updated)
  configs/model-selection/multilingual-e5-base.yaml: batch_size=32 (updated)
  configs/model-selection/phobert-base-v2.yaml: batch_size=32 (updated)
  configs/model-selection/phobert-base.yaml: batch_size=32 (updated)
  configs/model-selection/phobert-large.yaml: batch_size=32 (updated)
  configs/model-selection/vietnamese-bi-encoder.yaml: batch_size=32 (updated)


In [20]:
!python scripts/run_pipeline.py \
    configs/model-selection/*.yaml --results \
    results/model-selection/scores.json


Running: zero-shot-embeddinggemma-300m
  Queries: 26714  |  Corpus: 19962
Loading weights: 100% 314/314 [00:00<00:00, 1166.61it/s, Materializing param=norm.weight]                                
Batches: 100% 624/624 [04:52<00:00,  2.13it/s]
Batches: 100% 835/835 [00:46<00:00, 17.77it/s]
  mrr@1: 0.5658
  mrr@5: 0.6622
  mrr@10: 0.6700
  mrr@100: 0.6745
  success@1: 0.5658
  success@5: 0.8089
  success@10: 0.8664
  success@100: 0.9654
  recall@10: 0.8052
  recall@50: 0.8875
  recall@100: 0.9069
  precision@10: 0.0915
  precision@100: 0.0106
  f1@10: 0.1627
  f1@100: 0.0209
Batches: 100% 835/835 [00:46<00:00, 17.84it/s]
  Predictions → results/model-selection/embeddinggemma-300m/outputeval.txt
  Scores → results/model-selection/embeddinggemma-300m/scores.json

Running: zero-shot-F2LLM-v2-160M
  Queries: 26714  |  Corpus: 19962
modules.json: 100% 385/385 [00:00<00:00, 2.36MB/s]
config_sentence_transformers.json: 100% 224/224 [00:00<00:00, 1.23MB/s]
README.md: 100% 7.67k/7.67k [00:00<00

In [ ]:
!cat results/model-selection/scores.json | python3 -m json.tool

{
    "zero-shot-bge-m3": {
        "mrr@1": 0.5999475930223853,
        "mrr@5": 0.6912299418532211,
        "mrr@10": 0.6981240886711888,
        "mrr@100": 0.7021705056694094,
        "success@1": 0.5999475930223853,
        "success@5": 0.8290783858650894,
        "success@10": 0.8795762521524294,
        "success@100": 0.9682937785430861,
        "recall@10": 0.8173537506639992,
        "recall@50": 0.8915596839419562,
        "recall@100": 0.9093840872926755,
        "precision@10": 0.09320580968776483,
        "precision@100": 0.010605300591446883,
        "f1@10": 0.1656548086847328,
        "f1@100": 0.020932647828212934
    },
    "zero-shot-embeddinggemma-300m": {
        "mrr@1": 0.5658081904619301,
        "mrr@5": 0.6621765116917611,
        "mrr@10": 0.6699780360099034,
        "mrr@100": 0.6744728994190997,
        "success@1": 0.5658081904619301,
        "success@5": 0.8088642659279779,
        "success@10": 0.8664370741933068,
        "success@100": 0.9653739612188366

In [ ]:
import json
import pandas as pd

with open("results/model-selection/scores.json", "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame.from_dict(data, orient="index")
df.index.name = "experiment"

df.to_csv("results/model-selection/scores.csv")